In [25]:
!pip install dbt-duckdb

In [ ]:
# !pip install Faker
# !pip install pandas
!pip install dbt-snowflake

In [15]:
import datetime
from datetime import timedelta
from random import sample
import random
random.seed(42)
start_date = datetime.date(2026,6,11)
end_date = start_date+timedelta(days=90)

start = start_date
random_90_dates = [start_date]

while start_date != end_date:
    start_date += timedelta(days=1)
    random_90_dates.append(start_date)

# random K dates from pack
random_50_dates = sample(random_90_dates, k=50)
random_50_dates.sort()
match_dates = random_50_dates

In [16]:
import pandas as pd
import random
import datetime
##table1
# match id
# home team_id
# away team_id
# stadium id
# match date

team_rows = []
stadium_rows = []
match_rows = []
team_ids = ["T01","T02","T03","T04","T05","T06","T07","T08","T09","T10"]
team_name = ["France","Senegal","South Africa","Spain","Switzerland","Argentina","Uruguay","England",
"Croatia", "Belgium"]

for i in range(len(team_ids)):
    team_row = {
        "team_id": team_ids[i],
        "team_name": team_name[i]
        }
    team_rows.append(team_row)
    
team_table = pd.DataFrame.from_records(team_rows)
# print(team_table)

###table2
stadium_ids = ["S01","S02","S03","S04","S05","S06","S07","S08","S09","S10"]
stadium_names = ["Lumen Field","BC Place","MetLife Stadium",
"Mercedes-Benz Stadium","BMO Field","AT&T Stadium","NRG Stadium","GEHA Field at Arrowhead Stadium",
"SoFi Stadium","Hard Rock Stadium","Levi’s Stadium","Lincoln Financial Field"]
stadium_capacity = [random.randint(50,70)*1000 for i in range(10)]
avg_ticket_price = [random.randint(500,1000) for i in range(10)]

for i in range(len(stadium_ids)):
    stadium_row = {
        "stadium_id": stadium_ids[i],
        "stadium_name": stadium_names[i],
        "stadium_capacity": stadium_capacity[i],
        "avg_ticket_price": avg_ticket_price[i]
    }
    stadium_rows.append(stadium_row)

stadium_table = pd.DataFrame.from_records(stadium_rows)
# print(stadium_table)

match_ids = [str(i) for i in range(1,51)]
match_ids = list(map(lambda x: "M"+ x, match_ids))

for i in range(len(match_ids)):
    home = random.choice(team_ids)
    away = random.choice([t for t in team_ids if t != home])

    match_row = {
        "match_id": match_ids[i],
        "home_team_id": home,
        "away_team_id": away,
        "stadium_id": random.choice(stadium_ids),
        "match_date": match_dates[i]
    }
    match_rows.append(match_row)
match_table = pd.DataFrame.from_records(match_rows)
# print(match_table)


In [17]:
from datetime import datetime

def get_purchase_time(match_date):
    purchase_time = datetime.combine(
        match_date,
        datetime.min.time()
    ) - timedelta(
            days=random.randint(1, 60),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds = random.randint(0,59)
        )
    return purchase_time

In [18]:
###Transactional_data
## Purchase time and date
### amount

## match id
## user_id
### user_name (faker)
### user_email (faker)
## Quantity
### Ticket_ID
quantity_choices = [x for x in range(1,10)]
transactions = []
import uuid
from faker import Faker
Faker.seed(42)
fake = Faker()
total_transactions_num = 50000
ticket_categories_map = {'GA':1,'Student':0.7,'VIP':1.3}


for i in range(total_transactions_num):
    match_id = random.choice(match_ids)
    stadium_id = match_table[match_table["match_id"]==match_id]['stadium_id'].iloc[0]
    stadium_name = stadium_table[stadium_table["stadium_id"]==stadium_id]['stadium_name'].iloc[0]
    quantity = random.choice(quantity_choices)
    ticket_category = random.choice(list(ticket_categories_map.keys()))
    ticket_multiplier = ticket_categories_map[ticket_category]
    amount = stadium_table[stadium_table["stadium_id"]==stadium_id]['avg_ticket_price'].iloc[0]*quantity*ticket_multiplier
    match_date = match_table[match_table["match_id"]==match_id]['match_date'].iloc[0]
    purchase_time = get_purchase_time(match_date)

    
    ticketId = str(uuid.uuid4())
    
    user_id  = "FIFAW26"+str(i).zfill(6)
    user_name = fake.first_name() + " " + fake.last_name()
    user_email = fake.unique.ascii_free_email()
    
    transactions.append({
        "ticket_id": ticketId,
        "match_id": match_id,
        "user_id": user_id,
        "user_name": user_name,
        "user_email": user_email,
        "ticket_type": ticket_category,
        "quantity": quantity,
        "stadium": stadium_name,
        "price": round(amount, 2),
        "purchase_timestamp": purchase_time
    })

base_df = pd.DataFrame(transactions)
base_df.head()

,ticket_id,match_id,user_id,user_name,user_email,ticket_type,quantity,stadium,price,purchase_timestamp
0,daa41d17-81c6-45c2-b492-23e824c06345,M23,FIFAW26000000,Danielle Johnson,john21@hotmail.com,Student,7,GEHA Field at Arrowhead Stadium,3400.6,2026-06-14 00:56:17
1,35ca2e28-4bed-4a6d-aa5f-eabbd3e535d5,M42,FIFAW26000001,Joy Gardner,fjohnson@gmail.com,GA,2,Hard Rock Stadium,1464.0,2026-08-03 00:38:09
2,37025406-367a-465b-899e-87dca93506ac,M7,FIFAW26000002,Jesse Guzman,jennifermiles@yahoo.com,GA,4,AT&T Stadium,3772.0,2026-06-04 06:31:52
3,09260e93-803e-4a08-94db-6610790608ff,M28,FIFAW26000003,Jeffrey Lawrence,blakeerik@yahoo.com,Student,3,SoFi Stadium,1348.2,2026-06-30 16:04:01
4,526796b4-798f-4528-975d-9a6b1049e2e2,M5,FIFAW26000004,Matthew Moore,curtis61@yahoo.com,VIP,8,SoFi Stadium,6676.8,2026-06-08 22:18:26


In [19]:
base_df.to_csv('transaction_data.csv')

In [20]:
def generate_ingest_time(purchase_time):
    r = random.random()
    if r < 0.01:
        return purchase_time + timedelta(days=random.randint(1, 3))  # very late
    elif r < 0.11:
        return purchase_time + timedelta(hours=random.randint(1, 12))  # late
    else:
        return purchase_time + timedelta(minutes=random.randint(1, 30))  # normal


In [21]:
base_df["ingest_timestamp"] = base_df["purchase_timestamp"].apply(generate_ingest_time)
base_df.head()

,ticket_id,match_id,user_id,user_name,user_email,ticket_type,quantity,stadium,price,purchase_timestamp,ingest_timestamp
0,daa41d17-81c6-45c2-b492-23e824c06345,M23,FIFAW26000000,Danielle Johnson,john21@hotmail.com,Student,7,GEHA Field at Arrowhead Stadium,3400.6,2026-06-14 00:56:17,2026-06-14 01:05:17
1,35ca2e28-4bed-4a6d-aa5f-eabbd3e535d5,M42,FIFAW26000001,Joy Gardner,fjohnson@gmail.com,GA,2,Hard Rock Stadium,1464.0,2026-08-03 00:38:09,2026-08-03 00:47:09
2,37025406-367a-465b-899e-87dca93506ac,M7,FIFAW26000002,Jesse Guzman,jennifermiles@yahoo.com,GA,4,AT&T Stadium,3772.0,2026-06-04 06:31:52,2026-06-04 06:37:52
3,09260e93-803e-4a08-94db-6610790608ff,M28,FIFAW26000003,Jeffrey Lawrence,blakeerik@yahoo.com,Student,3,SoFi Stadium,1348.2,2026-06-30 16:04:01,2026-06-30 16:27:01
4,526796b4-798f-4528-975d-9a6b1049e2e2,M5,FIFAW26000004,Matthew Moore,curtis61@yahoo.com,VIP,8,SoFi Stadium,6676.8,2026-06-08 22:18:26,2026-06-08 22:22:26


In [22]:
num_duplicates = int(0.05 * len(base_df))
dup_rows = base_df.sample(num_duplicates)

duplicates = []

for _, row in dup_rows.iterrows():
    for _ in range(random.randint(1, 2)):
        dup = row.copy()
        dup["ingest_timestamp"] += timedelta(minutes=random.randint(5, 60))
        duplicates.append(dup)

dup_df = pd.DataFrame(duplicates)



In [23]:
messy_transactions = pd.concat([base_df, dup_df], ignore_index=True)
messy_transactions = messy_transactions.sample(frac=1).reset_index(drop=True)
messy_transactions.to_csv(
    "ticket_purchases_raw.csv",
    index=False
)

In [24]:
match_table.to_csv(
    "matches_data.csv",
    index=False
)
stadium_table.to_csv(
    "stadium_data.csv",
    index=False
)
team_table.to_csv(
    "team_table.csv",
    index=False
)

In [ ]:
import duckdb

con = duckdb.connect("dev.duckdb")
print(con)
print(con.execute("SHOW TABLES").fetchall())
# print(con.execute("SHOW SCHEMAS").fetchall())

[]


CatalogException: Catalog Error: Table with name SCHEMAS does not exist!

LINE 1: SHOW SCHEMAS
             ^

In [1]:
import duckdb

con = duckdb.connect("/Users/pavankumar_s/Desktop/Data Engineering/real_time_tickets_observability/dev.duckdb")

print(con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
""").fetchall())
import duckdb


print(
    con.execute("""
        SELECT *
        FROM dev.fct_daily_sales
        ORDER BY sale_date DESC, total_revenue DESC
        LIMIT 10
    """).fetchall()
)


# fct_daily_sales
con.close()

[('main', 'my_first_dbt_model'), ('main', 'stg_matches'), ('main', 'stg_stadiums'), ('main', 'stg_teams'), ('main', 'stg_ticket_purchases'), ('main', 'fct_daily_sales'), ('main', 'my_second_dbt_model')]
[(datetime.date(2026, 9, 7), datetime.date(2026, 9, 9), 'Senegal', 'South Africa', 'NRG Stadium', 17, 81, 43639.2, 2567.0117647058823, 23, 32, 26, 1076.4705882352941, 1740.0, datetime.datetime(2025, 12, 23, 21, 28, 23, 278744, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>)), (datetime.date(2026, 9, 6), datetime.date(2026, 9, 9), 'Senegal', 'South Africa', 'NRG Stadium', 16, 70, 39561.8, 2472.6125, 14, 25, 31, 9543.75, 86400.0, datetime.datetime(2025, 12, 23, 21, 28, 23, 278744, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>)), (datetime.date(2026, 9, 5), datetime.date(2026, 9, 7), 'England', 'Uruguay', 'Mercedes-Benz Stadium', 23, 135, 125651.70000000001, 5463.1173913043485, 14, 62, 59, 5522.608695652174, 39600.0, datetime.datetime(2025, 12, 23, 21, 